In [1]:
import socket
import time

# ESP32 Access Point IP and Port
ESP32_IP = "192.168.4.1"  # Default IP for ESP32 in AP mode
PORT = 80

In [2]:
with open('raw_audio.pcm', "wb") as f:
    for i in range(50):  # Adjust this to capture more/less data
        client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            # Connect to ESP32
            client_socket.connect((ESP32_IP, PORT))
            print(f"Connected to ESP32 at {ESP32_IP}:{PORT}")

            while True:
                data = client_socket.recv(1024)  # ✅ Read as binary
                if not data:
                    break
                
                f.write(data)  # ✅ Save raw PCM data

        except Exception as e:
            print(f"Error: {e}")
        finally:
            client_socket.close()
        
        time.sleep(0.002)  # Short delay between requests

print("Data saved to raw_audio.pcm")

Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
Connected to ESP32 at 192.168.4.1:80
C

In [3]:
import numpy as np
import soundfile as sf

pcm_file = "raw_audio.pcm"
sr = 16000  

audio_data = np.fromfile(pcm_file, dtype=np.int16)
gain=40

audio_data = audio_data * gain

sf.write("output.wav", audio_data, sr)

print("WAV file saved successfully!")

WAV file saved successfully!


In [4]:
audio_data.max()

6880

In [5]:
import requests

url = "http://127.0.0.1:8000/transcribe/"

file_path = "output.wav"

with open(file_path, "rb") as f:
    files = {"file": f}
    response = requests.post(url, files=files)

print(response.json())

{'transcript': ' Turn on the light.', 'device': 'cuda'}


In [6]:
command = response.json()['transcript']

print(command)

 Turn on the light.


In [7]:
command = command.split()

print(command)

['Turn', 'on', 'the', 'light.']


In [8]:
import time
import serial

ESP32_PORT = "COM8"  

# Open Serial Connection
ser = serial.Serial(ESP32_PORT, 9600, timeout=1)
time.sleep(2)  # Allow time for ESP32 to initialize

# Function to send command
def send_command(command):
    ser.write((command + "\n").encode())  # Send command
    time.sleep(1)  # Wait for ESP32 response
    response = ser.readline().decode().strip()  # Read response
    print(f"ESP32: {response}")

# Example Commands
for _ in range(2):
    try:
        send_command("light on")
    except:
        print('could not light on')
    try:
        send_command("light off")
    except:
        print('could not light off')

could not light on
ESP32: Received: light on
ESP32: Light ON
ESP32: Received: light off


In [9]:
send_command('light on')

ESP32: Light OFF


In [10]:
send_command('light off')

ESP32: Received: light on


In [12]:
if 'lights' in command or 'light' or 'light.' or 'light ' or 'light?' in command and 'on' in command:
    send_command('light on')
elif 'lights' in command or 'light' or 'light.' or 'light ' or 'light?' in command and 'off' in command:
    send_command('light off')

ESP32: Light ON


In [13]:
send_command('light off')

ESP32: Received: light off


In [15]:
ser.close()